# Render Video Remotion di Google Colab (T4 GPU)

Notebook ini dikonfigurasi khusus untuk render full video:
1. **Sejarah China & Tembok Besar**: Komposisi `SejarahChina`, resolusi 1920x1080 @ 30fps, 19.225 frame (~10,6 menit), 251 scene.
2. **Stickman Piramida**: Komposisi `PiramidaStickman`, resolusi 1920x1080 @ 30fps, 15.000 frame (~8,33 menit), 151 scene.

### Langkah Penggunaan:
1. Pastikan runtime Colab menggunakan **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar hasil video otomatis tersimpan permanen di Drive Anda (`remotion-out/`).
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi Chromium, Node.js 20, dan clone/update repo.
4. Pilih render:
   - Jalankan **Sel 3** untuk merender **Sejarah China** (Full).
   - Jalankan **Sel 4** untuk merender **Stickman Piramida** (Full).
   - ATAU jalankan **Sel 5** untuk merender **Kedua Video Sekaligus (Batch)** secara berurutan.
5. Jalankan **Sel 6** untuk mengunduh file `.mp4` langsung ke laptop Anda.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/remotion-out'
os.makedirs(out_dir, exist_ok=True)
print('Google Drive terhubung! Output video akan disimpan di:', out_dir)


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap merender SejarahChina dan PiramidaStickman."


In [ ]:
# Sel 3: Render Full Sejarah China (1920x1080 @ 30fps, 19.225 frame, ~10.6 menit)
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)

out_path = f'{out_dir}/sejarah-china-1080p.mp4'
print(f"Memulai Render Sejarah China -> {out_path}")
colab_render.run_render('SejarahChina', out_path, cwd='/content/vidio')


In [ ]:
# Sel 4: Render Full Stickman Piramida (1920x1080 @ 30fps, 15.000 frame, ~8.33 menit)
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)

out_path = f'{out_dir}/stickman-piramida-1080p.mp4'
print(f"Memulai Render Stickman Piramida -> {out_path}")
colab_render.run_render('PiramidaStickman', out_path, cwd='/content/vidio')


In [ ]:
# Sel 5: Render KEDUA Video Sekaligus Secara Otomatis (Batch Full Render)
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)

tasks = [
    ('SejarahChina', f'{out_dir}/sejarah-china-1080p.mp4'),
    ('PiramidaStickman', f'{out_dir}/stickman-piramida-1080p.mp4'),
]

for composition, out_path in tasks:
    print('\n' + '=' * 60)
    print(f'Memulai Render Batch: {composition} -> {out_path}')
    print('=' * 60)
    colab_render.run_render(composition, out_path, cwd='/content/vidio')

print('\nSELURUH RENDER SELESAI! Kedua video telah tersimpan rapi di Google Drive.')


In [ ]:
# Sel 6: Download Hasil Video ke Komputer/Laptop
from google.colab import files
import os

targets = [
    ('Sejarah China', '/content/drive/MyDrive/remotion-out/sejarah-china-1080p.mp4', '/content/remotion-out/sejarah-china-1080p.mp4'),
    ('Stickman Piramida', '/content/drive/MyDrive/remotion-out/stickman-piramida-1080p.mp4', '/content/remotion-out/stickman-piramida-1080p.mp4'),
]

found_any = False
for label, p_drive, p_local in targets:
    target_file = p_drive if os.path.exists(p_drive) else (p_local if os.path.exists(p_local) else None)
    if target_file:
        found_any = True
        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        print(f'Mengunduh {label} ({size_mb:.1f} MB) dari {target_file}...')
        files.download(target_file)

if not found_any:
    print('Belum ada file video yang ditemukan. Jalankan sel render terlebih dahulu.')
